# Einfach Deutsch — EDA & Error Analysis

This notebook is a template for exploring the processed dataset and analysing model errors.
Run each cell after installing the project dependencies (`pip install -r requirements.txt`).

In [ ]:
import sys
from pathlib import Path

# Make src/ importable from notebooks/
sys.path.insert(0, str(Path("..")))

import pandas as pd
from datasets import load_from_disk

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    import plotly.express as px
except ImportError:
    px = None

from src.evaluation.readability import lix_score, wstf_score
from src.utils.config import load_config

## 1. Load processed dataset

In [ ]:
config = load_config("../configs/config.yaml")
processed_dir = Path(config["paths"]["data_processed"])

# HuggingFace DatasetDict with train/validation/test splits
dataset = load_from_disk(str(processed_dir / "parallel"))
df = dataset["train"].to_pandas()
df.head()

## 2. Distribution of length ratios

In [ ]:
df["len_complex"] = df["complex"].astype(str).str.split().str.len()
df["len_simple"] = df["simple"].astype(str).str.split().str.len()
df["length_ratio"] = df["len_simple"] / df["len_complex"].replace(0, 1)

if px is not None:
    fig = px.histogram(df, x="length_ratio", nbins=50, title="Length ratio (simple / complex)")
    fig.show()
elif plt is not None:
    plt.figure(figsize=(8, 4))
    plt.hist(df["length_ratio"], bins=50, edgecolor="black")
    plt.title("Length ratio (simple / complex)")
    plt.xlabel("ratio")
    plt.ylabel("count")
    plt.show()
else:
    print("Install matplotlib or plotly to plot.")

## 3. LIX before/after histogram

In [ ]:
df["lix_complex"] = df["complex"].astype(str).apply(lix_score)
df["lix_simple"] = df["simple"].astype(str).apply(lix_score)

if px is not None:
    fig = px.histogram(
        df,
        x=["lix_complex", "lix_simple"],
        nbins=50,
        title="LIX distribution before and after simplification",
        labels={"value": "LIX", "variable": "version"},
    )
    fig.show()
elif plt is not None:
    plt.figure(figsize=(8, 4))
    plt.hist(df["lix_complex"], bins=50, alpha=0.6, label="complex")
    plt.hist(df["lix_simple"], bins=50, alpha=0.6, label="simple")
    plt.title("LIX distribution before and after simplification")
    plt.xlabel("LIX")
    plt.ylabel("count")
    plt.legend()
    plt.show()
else:
    print("Install matplotlib or plotly to plot.")

## 4. Error analysis example

Inspect cases where the model changed meaning, lost entities, or did not simplify enough.

In [ ]:
# Load a CSV with source/reference/prediction columns from an evaluation run
eval_path = Path("../outputs/evaluation/results.csv")
if eval_path.exists():
    eval_df = pd.read_csv(eval_path)
    # Show rows with the largest readability *increase* (bad simplifications)
    eval_df["lix_delta"] = eval_df["lix_after"] - eval_df["lix_before"]
    bad_cases = eval_df.nlargest(5, "lix_delta")
    display(bad_cases[["source", "prediction", "reference", "sari", "lix_delta"]])
else:
    print(f"Evaluation results not found at {eval_path}. Run evaluate.py first.")